# DICOM to Key-Frame Extraction and Angiogram Preprocessing Pipeline

 Overview

This notebook implements the **DICOM → Key Frame → Angiogram Preprocessing** pipeline
designed for the *AI-Driven Coronary Disease Detection and Decision Support System*.

The goal is to:
- Convert angiogram DICOM files into individual frames
- Automatically select diagnostically relevant key frames
- Apply standardized angiogram preprocessing
- Prepare outputs for the blockage detection and localization module


## Step 1: Upload Angiogram (DICOM)

In [1]:
from pathlib import Path

# Path to the uploaded DICOM file
dicom_path = Path("sample_angiogram.dcm")

print(f"DICOM file selected: {dicom_path}")

DICOM file selected: sample_angiogram.dcm


The system accepts angiogram data in **DICOM format**, which is the standard
medical imaging format used in hospitals. This file may contain multiple frames
representing a complete angiography sequence.

## Step 2: Validate DICOM Format

In [2]:
import pydicom

def validate_dicom(file_path: Path) -> bool:
    """
    Validates whether the given file is a readable DICOM file.
    """
    try:
        pydicom.dcmread(file_path, stop_before_pixels=True)
        return True
    except Exception as e:
        print(f"Invalid DICOM file: {e}")
        return False


# Validate input file
is_valid = validate_dicom(dicom_path)
print("DICOM validation result:", is_valid)

Invalid DICOM file: [Errno 2] No such file or directory: 'sample_angiogram.dcm'
DICOM validation result: False


Before processing, the system validates whether the uploaded file is a valid
DICOM file. This prevents corrupted or unsupported files from entering the
pipeline.

## Step 3: Read DICOM Metadata & Frames

In [3]:
import numpy as np
import pydicom
from pathlib import Path

def read_dicom_or_simulate(file_path: Path):
    """
    Reads a DICOM file if available.
    If not, generates simulated angiogram frames for development/testing.
    """

    if file_path.exists():
        ds = pydicom.dcmread(file_path)

        metadata = {
            "Source": "DICOM",
            "Modality": ds.get("Modality", "Unknown"),
            "Rows": ds.get("Rows", None),
            "Columns": ds.get("Columns", None),
            "NumberOfFrames": ds.get("NumberOfFrames", 1)
        }

        frames = ds.pixel_array.astype(np.float32)

    else:
        # ---- SIMULATION MODE ----
        print("DICOM file not found. Using simulated angiogram frames.")

        num_frames = 20
        height, width = 512, 512

        frames = np.random.normal(
            loc=100, scale=25, size=(num_frames, height, width)
        ).astype(np.float32)

        metadata = {
            "Source": "Simulated",
            "Modality": "XA",
            "Rows": height,
            "Columns": width,
            "NumberOfFrames": num_frames
        }

    return metadata, frames


metadata, frames = read_dicom_or_simulate(dicom_path)

print("Metadata:", metadata)
print("Frames shape:", frames.shape)

DICOM file not found. Using simulated angiogram frames.
Metadata: {'Source': 'Simulated', 'Modality': 'XA', 'Rows': 512, 'Columns': 512, 'NumberOfFrames': 20}
Frames shape: (20, 512, 512)


## Step 3: Read DICOM Metadata and Frames



Once validated, the DICOM file is read to extract:
- Image metadata (useful for traceability)
- Multi-frame pixel data representing the angiogram sequence

## Step 4: Extract Individual Frames

In [4]:
def extract_frames(frame_array: np.ndarray):
    """
    Splits multi-frame DICOM pixel array into individual frames.
    """
    return [frame_array[i] for i in range(frame_array.shape[0])]


frame_list = extract_frames(frames)

print(f"Total frames extracted: {len(frame_list)}")

Total frames extracted: 20


## Step 05: Frame Quality Analysis and Diagnostic Scoring

In this step, **each extracted frame is evaluated using quantitative image-quality metrics**
to determine how diagnostically informative it is. These metrics are later combined
into a single **diagnostic score**, which is used for key-frame selection.

### Step 5.1: Metric Computation Function

In [5]:
import cv2
import numpy as np

def compute_frame_quality_metrics(frame: np.ndarray):
    """
    Computes quality metrics for a single angiogram frame.
    """
    # Ensure grayscale (angiograms are usually grayscale already)
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()

    # --- Mean Intensity ---
    mean_intensity = np.mean(gray)

    # --- Contrast (Standard Deviation) ---
    contrast = np.std(gray)

    # --- Edge Strength (Sobel Gradient Magnitude) ---
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_strength = np.mean(np.sqrt(sobel_x**2 + sobel_y**2))

    # --- Noise Estimation (High-frequency variation) ---
    noise = np.std(gray - cv2.GaussianBlur(gray, (5, 5), 0))

    return {
        "mean_intensity": mean_intensity,
        "contrast": contrast,
        "edge_strength": edge_strength,
        "noise": noise
    }


### Step 5.2: Apply Metrics to All Frames

In [6]:
frame_metrics = []

for idx, frame in enumerate(frame_list):
    metrics = compute_frame_quality_metrics(frame)
    metrics["frame_index"] = idx
    frame_metrics.append(metrics)

frame_metrics[:3]  # Preview first few frames


[{'mean_intensity': np.float32(99.934074),
  'contrast': np.float32(24.995155),
  'edge_strength': np.float64(108.23669707326609),
  'noise': np.float32(22.266886),
  'frame_index': 0},
 {'mean_intensity': np.float32(100.04976),
  'contrast': np.float32(24.99355),
  'edge_strength': np.float64(108.42398596198605),
  'noise': np.float32(22.25544),
  'frame_index': 1},
 {'mean_intensity': np.float32(99.97723),
  'contrast': np.float32(24.975128),
  'edge_strength': np.float64(108.31290727146066),
  'noise': np.float32(22.24706),
  'frame_index': 2}]

### Step 5.3: Diagnostic Score Computation

In [7]:
def compute_diagnostic_score(metrics, w1=0.3, w2=0.3, w3=0.3, w4=0.1):
    """
    Computes a diagnostic score from frame quality metrics.
    """
    score = (
        w1 * metrics["mean_intensity"]
        + w2 * metrics["contrast"]
        + w3 * metrics["edge_strength"]
        - w4 * metrics["noise"]
    )
    return score


for m in frame_metrics:
    m["diagnostic_score"] = compute_diagnostic_score(m)


### Diagnostic Scoring Formulation

Each frame is assigned a diagnostic score based on a weighted combination
of the computed quality metrics.

Let:

- $( \mu $) = Mean intensity  
- $( \sigma $) = Contrast (standard deviation)  
- $( E $) = Edge strength  
- $( N $) = Noise estimate  

The diagnostic score $( S $) for a frame is defined as:

$[
S = w_1 \cdot \mu + w_2 \cdot \sigma + w_3 \cdot E - w_4 \cdot N
$]

Where:
- $( w_1, w_2, w_3, w_4 $) are empirically chosen weights
- Noise is subtracted to penalize low-quality frames

This formulation prioritizes frames with:
- Clear dye diffusion
- High vessel contrast
- Strong vessel edges
- Minimal noise


Unlike processing all frames equally, this approach ensures that:
- Redundant or low-quality frames are discarded
- Only diagnostically meaningful frames are selected
- Computational cost is reduced
- Downstream blockage detection models receive higher-quality inputs

This step significantly improves robustness and aligns the system with
real-world clinical angiogram interpretation.

## Step 06: Key Frame Selection

The selection process follows three principles:

1. **Ranking** – Frames are sorted by diagnostic score (highest first)
2. **Top-K Selection** – Only the best 3–5 frames are retained
3. **Temporal Spacing** – Frames too close in time are avoided to reduce redundancy

This ensures that the final set of frames is:
- Informative
- Diverse
- Computationally efficient

### Step 6.1: Convert Metrics to Structured Array

In [8]:
import pandas as pd

# Convert frame metrics to DataFrame for easier ranking
metrics_df = pd.DataFrame(frame_metrics)

# Sort frames by diagnostic score (descending)
metrics_df = metrics_df.sort_values(
    by="diagnostic_score", ascending=False
).reset_index(drop=True)

metrics_df.head()


,mean_intensity,contrast,edge_strength,noise,frame_index,diagnostic_score
0,100.029442,25.068754,108.844275,22.338757,6,67.948868
1,100.087761,24.992702,108.462270,22.265659,8,67.836254
2,99.986969,24.998297,108.545986,22.257727,15,67.833606
3,100.044937,25.022186,108.462376,22.295801,9,67.829270
4,100.011032,24.997656,108.500123,22.249540,13,67.827692


### Step 6.2: Temporal Spacing Helper Function

In [9]:
def select_top_k_with_spacing(
    df,
    k=5,
    min_frame_gap=3
):
    """
    Selects top-K frames while enforcing minimum temporal spacing.
    
    Parameters:
    - df: DataFrame sorted by diagnostic score
    - k: number of frames to select
    - min_frame_gap: minimum frame index gap to avoid redundancy
    """
    selected_frames = []
    selected_indices = []

    for _, row in df.iterrows():
        frame_idx = row["frame_index"]

        # Check temporal spacing
        if all(abs(frame_idx - idx) >= min_frame_gap for idx in selected_indices):
            selected_frames.append(row)
            selected_indices.append(frame_idx)

        if len(selected_frames) == k:
            break

    return pd.DataFrame(selected_frames)


### Step 6.3: Select Key Diagnostic Frames

In [10]:
# Select top 3–5 key frames
K = 5
MIN_FRAME_GAP = 3

key_frames_df = select_top_k_with_spacing(
    metrics_df,
    k=K,
    min_frame_gap=MIN_FRAME_GAP
)

key_frames_df


,mean_intensity,contrast,edge_strength,noise,frame_index,diagnostic_score
0,100.029442,25.068754,108.844275,22.338757,6.0,67.948868
2,99.986969,24.998297,108.545986,22.257727,15.0,67.833606
3,100.044937,25.022186,108.462376,22.295801,9.0,67.829270
5,100.049759,24.993549,108.423986,22.255440,1.0,67.814645
8,100.021782,24.995504,108.418184,22.273264,18.0,67.803317


### Formal Interpretation

Let $( F = \{f_1, f_2, \dots, f_n\} )$ be the set of all angiogram frames  
and \( S(f_i) \) be the diagnostic score of frame $( f_i )$.

The selected key-frame set $( K \subset F )$ satisfies:

$$K = \arg\max_{|K| \leq k} \sum_{f \in K} S(f)$$

subject to:

$[
| index(f_i) - index(f_j) | \geq d \quad \forall f_i, f_j \in K
]$

where:
- $( k )$ is the maximum number of frames (3–5)
- $( d )$ is the minimum temporal spacing constraint

This ensures optimal diagnostic coverage with minimal redundancy.
